<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_CBP_FineTuning_LLM_Mistral_7B_Instruct_v0_1_for_TEXT_to_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://medium.com/thedeephub/fine-tuning-the-llm-mistral-7b-for-text-to-sql-with-sql-create-context-dataset-4e9234f7691c

## SETUP

In [ ]:
!pip install "kernels==0.17.2"

In [ ]:
!pip show transformers torch trl peft accelerate datasets | egrep "Name|Version"

In [1]:
!nvidia-smi

Thu Nov 19 01:56:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   35C    P0             90W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## TRAIN TOPO-CBP

In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # must be set before CUDA init
import math
import hashlib
import random
import torch
from torch.optim import Optimizer
from transformers import (AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
                          Trainer, DataCollatorForSeq2Seq, TrainerCallback)
from datasets import load_dataset

# =====================================================================
# 0. SEED
# =====================================================================
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 1. TOPO-2026 OPTIMIZER (full fine-tune; bf16 weights, grads, 2 moments, Kahan buffer: ~68 GiB)
# =====================================================================
SAFETY_CONSTANT = 0.85
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32, dst_bf16):
    """Write fp32 `src` into bf16 `dst` with stochastic rounding (unbiased), so updates
    smaller than one bf16 step are not systematically lost. Modifies `src` in place."""
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))   # random value below the bf16 cut-off
    bits.bitwise_and_(-65536)                          # truncate low 16 bits -> exactly bf16-representable
    dst_bf16.copy_(src_fp32)


class TopoAdamW(Optimizer):
    """AdamW for a bf16 model with bf16 moments and a bf16 Kahan compensation buffer. Arithmetic is done
    in fp32 per tensor; moments are stored with stochastic rounding and weights with error compensation.
    Also keeps selected input-embedding rows bit-identical."""

    def __init__(self, params, lr=1e-5, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2, embedding_layer=None, prime_anchors=None,
                 lambda_euler=SAFETY_CONSTANT):
        if embedding_layer is None:
            raise ValueError("'embedding_layer' must be passed explicitly.")
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

        self.embedding_layer = embedding_layer
        self.lambda_euler = lambda_euler  # kept for API compatibility; see note in step()
        w = embedding_layer.weight.data
        self.valid_anchors = [i for i in (prime_anchors or PRIME_ANCHORS) if i < w.shape[0]]
        self.baseline = {i: w[i].detach().clone() for i in self.valid_anchors}
        self.certified_sha256_digest = self._digest()
        print(f"[TopoAdamW] Baseline locked. Digest: {self.certified_sha256_digest[:16]}")

    def _digest(self):
        w = self.embedding_layer.weight.data
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight
        # The baseline was captured while the model was still on CPU; keep it on the weights' device.
        if self.baseline[self.valid_anchors[0]].device != emb_w.device:
            self.baseline = {i: t.to(emb_w.device) for i, t in self.baseline.items()}
        # Zero anchor-row grads so their moments stay zero.
        # (Scaling grads by lambda is a no-op under Adam, so it is not applied.)
        if emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p)      # bf16, 2 bytes/param
                    state["exp_avg_sq"] = torch.zeros_like(p)   # bf16, 2 bytes/param
                    state["comp"] = torch.zeros_like(p)         # bf16 Kahan compensation, 2 bytes/param
                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]

                # fp32 math on short-lived temporaries. Moments are stored back in bf16 with stochastic
                # rounding; the weight update uses Kahan compensation: the rounding remainder of each
                # step is carried into the next one, so small updates accumulate instead of being lost.
                g = p.grad.float()
                m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
                v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
                del g
                w = p.float()
                denom = (v.sqrt() / math.sqrt(1.0 - beta2 ** t)).add_(eps)
                y = m.div(denom).mul_(-lr / (1.0 - beta1 ** t))      # Adam step
                del denom
                y.add_(w, alpha=-lr * wd).add_(comp.float())         # decoupled weight decay + carried remainder
                _stochastic_round_(m, m_buf)
                _stochastic_round_(v, v_buf)
                del m, v
                total = w.add_(y)                                    # exact fp32 sum
                new = total.to(torch.bfloat16)                       # round to nearest bf16
                comp.copy_(total.sub_(new.float()))                  # remainder -> compensation buffer
                p.copy_(new)
                del y, w, total, new

                if p is emb_w:  # bit-exact restore of the locked anchor rows
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i])
                        comp[i].zero_()
        return loss

    def audit_telemetry(self):
        w = self.embedding_layer.weight.data
        drift = max(torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
                    for i in self.valid_anchors)
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


# =====================================================================
# 2. MODEL / TOKENIZER
# =====================================================================
model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="kernels-community/flash-attn2",  # needs: pip install "kernels==0.17.2"
    dtype=torch.bfloat16,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

topo_optimizer = TopoAdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-6,  # the scheduler takes its learning rate from here, not from TrainingArguments
    weight_decay=1e-2,
    embedding_layer=model.get_input_embeddings(),
    prime_anchors=PRIME_ANCHORS,
)

# =====================================================================
# 3. DATA (chat template + loss only on the answer)
# =====================================================================
system_message = ("You are a text-to-SQL query translator. Users will ask you questions in "
                  "English and you will generate a SQL query based on the provided SCHEMA.\n"
                  "SCHEMA:\n{schema}")


def preprocess_function(sample):
    # v0.1's template has no system role, so the schema goes into the user turn.
    user = system_message.format(schema=sample["context"]) + "\n\n" + sample["question"]
    # transformers 5: apply_chat_template(tokenize=True) returns a BatchEncoding, not a list.
    # Render to text first, then tokenize; "<s>" in the text is parsed as the BOS special token.
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user}], tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(sample["answer"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    input_ids = (list(prompt_ids) + answer_ids)[:1024]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:1024]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=SEED)
train_raw, eval_raw = raw.select(range(12500)), raw.select(range(12500, 13000))
train_ds = train_raw.map(preprocess_function, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(preprocess_function, remove_columns=eval_raw.column_names)

# =====================================================================
# 4. TRAINING
# =====================================================================
args = TrainingArguments(
    output_dir="Mistral-7B-text-to-sql-topo",
    num_train_epochs=3,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=16,  # effective batch 3 x 16 = 48 (was 6, too noisy for full fine-tuning)
    gradient_checkpointing=True,
    train_sampling_strategy="group_by_length",
    logging_steps=20,   # train loss printed every 20 steps (= 960 samples); 780 steps in total
    eval_strategy="steps",
    eval_steps=100,     # validation loss every 100 steps (the notebook table only adds rows at evals)
    save_strategy="epoch",
    save_total_limit=1,
    save_only_model=True,
    bf16=True,
    tf32=True,
    max_grad_norm=1.0,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    seed=SEED,
    data_seed=SEED,
    report_to="none",  # lowercase: the string "None" is rejected by transformers 5.22.1
    push_to_hub=False,  # set True and add hub_model_id when ready
)

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"step {state.global_step:>5} | train loss {logs['loss']:.4f} | "
                  f"grad norm {logs.get('grad_norm', float('nan')):.2f} | lr {logs.get('learning_rate', 0.0):.2e}",
                  flush=True)


trainer = Trainer(
    model=model,
    callbacks=[PrintLossCallback()],
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8,
                                         return_tensors="pt", padding=True),
    optimizers=(topo_optimizer, None),
)

print("[baseline] validation loss before training:", round(trainer.evaluate()["eval_loss"], 4), flush=True)
trainer.train()

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")

trainer.save_model()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[TopoAdamW] Baseline locked. Digest: 6e461fb9caea3446


Training Loss,Validation Loss,Step
No log,0.480923,0


[baseline] validation loss before training: 0.4809


... (6 outputs left out)

step   540 | train loss 0.0112 | grad norm 0.66 | lr 1.16e-06
step   560 | train loss 0.0066 | grad norm 0.84 | lr 9.90e-07
step   580 | train loss 0.0084 | grad norm 1.08 | lr 8.31e-07
step   600 | train loss 0.0064 | grad norm 0.81 | lr 6.84e-07
step   620 | train loss 0.0068 | grad norm 1.58 | lr 5.49e-07
step   640 | train loss 0.0090 | grad norm 0.70 | lr 4.27e-07
step   660 | train loss 0.0049 | grad norm 0.40 | lr 3.19e-07
step   680 | train loss 0.0100 | grad norm 0.95 | lr 2.26e-07
step   700 | train loss 0.0070 | grad norm 0.64 | lr 1.48e-07
step   720 | train loss 0.0057 | grad norm 1.38 | lr 8.63e-08
step   740 | train loss 0.0113 | grad norm 0.91 | lr 4.09e-08
step   760 | train loss 0.0051 | grad norm 0.59 | lr 1.22e-08
step   780 | train loss 0.0064 | grad norm 0.93 | lr 3.39e-10


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[TopoAudit] Max Anchor Drift: 0.000000e+00 | Hash Matched: True | Digest: 6e461fb9caea3446


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## EVAL

In [5]:
model.eval()
model.config.use_cache = True
hits = 0
samples = eval_raw.select(range(100))
for s in samples:
    user = system_message.format(schema=s["context"]) + "\n\n" + s["question"]
    text = tokenizer.apply_chat_template([{"role": "user", "content": user}], tokenize=False)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    pred = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    ok = " ".join(pred.lower().split()) == " ".join(s["answer"].lower().split())
    hits += ok
    print(f"Q: {s['question']}\n  expected: {s['answer']}\n  got:      {pred}\n  match: {ok}\n")
print(f"exact match: {hits}/{len(samples)}")

Q: How many copies per particle are created by the enterotoxin?
  expected: SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"
  got:      SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"
  match: True

Q: What is the Score of the game of the Player in Place 4?
  expected: SELECT score FROM table_name_16 WHERE place = "4"
  got:      SELECT score FROM table_name_16 WHERE place = 4
  match: False

Q: How many heats does Julia Wilkinson have?
  expected: SELECT COUNT(heat) FROM table_name_88 WHERE name = "julia wilkinson"
  got:      SELECT COUNT(heat) FROM table_name_88 WHERE name = "julia wilkinson"
  match: True

Q: Which Away captain has a Date of 2,3,4,5 january 1999?
  expected: SELECT away_captain FROM table_name_69 WHERE date = "2,3,4,5 january 1999"
  got:      SELECT away_captain FROM table_name_69 WHERE date = "2,3,4,5 january 1999"
  match: True

Q: Who's the player that's male and on the original season of RW: Key West?

In [6]:
import re
import numpy as np

# Optional semantic similarity setup (gracefully skips if sentence-transformers isn't installed)
try:
    from sentence_transformers import SentenceTransformer
    sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    USE_SEMANTIC = True
except ImportError:
    sim_model = None
    USE_SEMANTIC = False


def normalize_sql(sql: str) -> str:
    """Normalize whitespace, casing, and unquote numeric literals for cosmetic comparison."""
    s = " ".join(sql.strip().lower().split())
    s = re.sub(r'=\s*["\'](-?\d+(?:\.\d+)?)["\']', r"= \1", s)
    return s


def semantic_similarity(a: str, b: str) -> float | None:
    if not USE_SEMANTIC:
        return None
    emb = sim_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))


model.eval()
model.config.use_cache = True

exact_hits = 0
norm_hits = 0
sim_scores = []

samples = eval_raw.select(range(100))

for s in samples:
    user = system_message.format(schema=s["context"]) + "\n\n" + s["question"]
    text = tokenizer.apply_chat_template([{"role": "user", "content": user}], tokenize=False)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    pred = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    expected = s["answer"].strip()
    exact_ok = " ".join(pred.lower().split()) == " ".join(expected.lower().split())
    norm_ok = normalize_sql(pred) == normalize_sql(expected)
    sim = semantic_similarity(pred, expected)

    exact_hits += int(exact_ok)
    norm_hits += int(norm_ok)
    if sim is not None:
        sim_scores.append(sim)

    sim_str = f" | sim: {sim:.4f}" if sim is not None else ""
    print(
        f"Q: {s['question']}\n"
        f"  expected: {expected}\n"
        f"  got:      {pred}\n"
        f"  exact: {exact_ok} | norm: {norm_ok}{sim_str}\n"
    )

print(f"exact match:      {exact_hits}/{len(samples)}")
print(f"normalized match: {norm_hits}/{len(samples)}")
if sim_scores:
    print(f"mean semantic similarity: {np.mean(sim_scores):.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q: How many copies per particle are created by the enterotoxin?
  expected: SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"
  got:      SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"
  exact: True | norm: True | sim: 1.0000

Q: What is the Score of the game of the Player in Place 4?
  expected: SELECT score FROM table_name_16 WHERE place = "4"
  got:      SELECT score FROM table_name_16 WHERE place = 4
  exact: False | norm: True | sim: 0.9916

Q: How many heats does Julia Wilkinson have?
  expected: SELECT COUNT(heat) FROM table_name_88 WHERE name = "julia wilkinson"
  got:      SELECT COUNT(heat) FROM table_name_88 WHERE name = "julia wilkinson"
  exact: True | norm: True | sim: 1.0000

Q: Which Away captain has a Date of 2,3,4,5 january 1999?
  expected: SELECT away_captain FROM table_name_69 WHERE date = "2,3,4,5 january 1999"
  got:      SELECT away_captain FROM table_name_69 WHERE date = "2,3,4,5 january 1999"
  exact: 

In [4]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/mistral-7b-text-to-sql-topo"
model.config.use_cache = True
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!ls -lh {SAVE_DIR}

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

total 14G
-rw------- 1 root root 1.1K Nov 19 02:54 chat_template.jinja
-rw------- 1 root root  685 Nov 19 02:54 config.json
-rw------- 1 root root  132 Nov 19 02:54 generation_config.json
-rw------- 1 root root  14G Nov 19 02:54 model.safetensors
-rw------- 1 root root  493 Nov 19 02:54 tokenizer_config.json
-rw------- 1 root root 3.4M Nov 19 02:54 tokenizer.json


In [ ]:
!ls -lh /content/drive/MyDrive/topogpt/llama.cpp/

## GGUF

In [10]:
import os
import subprocess
import sys

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
MODEL_DIR = "/content/drive/MyDrive/mistral-7b-text-to-sql-topo"
OUT_DIR = "/content/gguf_out"

os.makedirs(OUT_DIR, exist_ok=True)

convert_script = os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")
out_file = os.path.join(OUT_DIR, "mistral-7b-text-to-sql-topo-bf16.gguf")

cmd = [
    sys.executable,
    convert_script,
    MODEL_DIR,
    "--outtype", "bf16",
    "--outfile", out_file,
]

subprocess.run(cmd, check=True)
print(f"Saved base GGUF to: {out_file}")

Saved base GGUF to: /content/gguf_out/mistral-7b-text-to-sql-topo-bf16.gguf


In [14]:
import os
import shutil
import subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
OUT_DIR = "/content/gguf_out"
DRIVE_DEST = "/content/drive/MyDrive/topogpt"

base_gguf = os.path.join(OUT_DIR, "mistral-7b-text-to-sql-topo-bf16.gguf")
bin_dir = os.path.join(LLAMA_CPP_DIR, "build", "bin")
quant_bin = os.path.join(bin_dir, "llama-quantize")

# Pass build/bin in LD_LIBRARY_PATH so libllama.so / libggml.so are resolved
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{bin_dir}:{env.get('LD_LIBRARY_PATH', '')}"

# Quick check: if the Drive binary still fails (e.g. stale CUDA libs), build cleanly in /content
test_run = subprocess.run([quant_bin, "--help"], env=env, capture_output=True, text=True)
if test_run.returncode != 0:
    print("Drive binary failed with:", test_run.stderr.strip())
    print("Building fresh CPU llama-quantize in /content/llama_build ...")
    local_build = "/content/llama_build"
    subprocess.run(
        ["cmake", "-S", LLAMA_CPP_DIR, "-B", local_build, "-DGGML_CUDA=OFF", "-DBUILD_SHARED_LIBS=OFF"],
        check=True,
    )
    subprocess.run(
        ["cmake", "--build", local_build, "--config", "Release", "-j", "--target", "llama-quantize"],
        check=True,
    )
    bin_dir = os.path.join(local_build, "bin")
    quant_bin = os.path.join(bin_dir, "llama-quantize")
    env["LD_LIBRARY_PATH"] = f"{bin_dir}:{env.get('LD_LIBRARY_PATH', '')}"

QUANT_TYPES = ["Q4_K_M", "Q8_0"]

for qtype in QUANT_TYPES:
    q_name = f"mistral-7b-text-to-sql-topo-{qtype}.gguf"
    q_path = os.path.join(OUT_DIR, q_name)
    subprocess.run([quant_bin, base_gguf, q_path, qtype], env=env, check=True)

    dest_path = os.path.join(DRIVE_DEST, q_name)
    shutil.copy2(q_path, dest_path)
    size_gb = os.path.getsize(dest_path) / (1024 ** 3)
    print(f"Saved {qtype} ({size_gb:.2f} GB) -> {dest_path}")

Drive binary failed with: 
Building fresh CPU llama-quantize in /content/llama_build ...
Saved Q4_K_M (4.07 GB) -> /content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q4_K_M.gguf
Saved Q8_0 (7.17 GB) -> /content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf


## GGUF-EVAL

In [15]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.6/893.6 MB 121.6 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [16]:
from llama_cpp import Llama

GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q4_K_M.gguf"
# Or swap to the 8-bit quant:
# GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf"

llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=1024,
    n_gpu_layers=-1,  # set to 0 for CPU-only inference
    verbose=False,
)

system_message = (
    "You are a text-to-SQL query translator. Users will ask you questions in "
    "English and you will generate a SQL query based on the provided SCHEMA.\n"
    "SCHEMA:\n{schema}"
)

test_cases = [
    {
        "schema": "CREATE TABLE head (age INTEGER, name VARCHAR)",
        "question": "What is the average age of the heads?",
    },
    {
        "schema": "CREATE TABLE instructor (name VARCHAR, salary INTEGER, dept_name VARCHAR)",
        "question": "Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.",
    },
]

for item in test_cases:
    user_content = system_message.format(schema=item["schema"]) + "\n\n" + item["question"]
    resp = llm.create_chat_completion(
        messages=[{"role": "user", "content": user_content}],
        max_tokens=128,
        temperature=0.0,
    )
    sql = resp["choices"][0]["message"]["content"].strip()
    print(f"Q:   {item['question']}\nSQL: {sql}\n")

Q:   What is the average age of the heads?
SQL: SELECT AVG(age) FROM head WHERE name = "Head"

Q:   Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.
SQL: SELECT name FROM instructor WHERE salary > (SELECT MIN(salary) FROM instructor WHERE dep_name = 'BioLogy')



In [17]:
from llama_cpp import Llama

MODELS = {
    "Q4_K_M": "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q4_K_M.gguf",
    "Q8_0": "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf",
}

system_message = (
    "You are a text-to-SQL query translator. Users will ask you questions in "
    "English and you will generate a SQL query based on the provided SCHEMA.\n"
    "SCHEMA:\n{schema}"
)

test_cases = [
    {
        "schema": "CREATE TABLE head (age INTEGER, name VARCHAR)",
        "question": "What is the average age of the heads?",
    },
    {
        "schema": "CREATE TABLE instructor (name VARCHAR, salary INTEGER, dept_name VARCHAR)",
        "question": "Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.",
    },
]


def format_mistral_prompt(schema: str, question: str) -> str:
    user_content = system_message.format(schema=schema) + "\n\n" + question
    # Matches Mistral-7B-Instruct-v0.1 single-turn [INST] layout
    return f"<s>[INST] {user_content} [/INST]"


for label, gguf_path in MODELS.items():
    print("=" * 72)
    print(f"Testing {label}: {gguf_path}")
    print("=" * 72)

    llm = Llama(
        model_path=gguf_path,
        n_ctx=1024,
        n_gpu_layers=-1,  # set to 0 for CPU-only inference
        verbose=False,
    )

    for item in test_cases:
        prompt = format_mistral_prompt(item["schema"], item["question"])
        out = llm(
            prompt,
            max_tokens=128,
            temperature=0.0,
            stop=["</s>"],
        )
        sql = out["choices"][0]["text"].strip()
        print(f"Q:   {item['question']}\nSQL: {sql}\n")

    # Free memory before loading the next quant
    del llm

Testing Q4_K_M: /content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q4_K_M.gguf


/usr/local/lib/python3.13/dist-packages/llama_cpp/llama.py:1314: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


Q:   What is the average age of the heads?
SQL: SELECT AVG(age) FROM head WHERE name = "Head"

Q:   Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.
SQL: SELECT name FROM instructor WHERE salary > (SELECT MIN(salary) FROM instructor WHERE dep_name = 'BioLogy')

Testing Q8_0: /content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf
Q:   What is the average age of the heads?
SQL: SELECT AVG(age) FROM head WHERE name = "Head"

Q:   Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.
SQL: SELECT name FROM instructor WHERE salary > (SELECT MIN(salary) FROM instructor WHERE dep_name = 'BioLogy')



In [18]:
import re
import numpy as np
from llama_cpp import Llama

GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q4_K_M.gguf"
# Swap to Q8_0 to benchmark the 8-bit model:
# GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf"

N_SAMPLES = 100

# Optional semantic similarity setup
try:
    from sentence_transformers import SentenceTransformer
    sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    USE_SEMANTIC = True
except ImportError:
    sim_model = None
    USE_SEMANTIC = False


def normalize_sql(sql: str) -> str:
    """Normalize whitespace, casing, and unquote numeric literals for cosmetic comparison."""
    s = " ".join(sql.strip().lower().split())
    s = re.sub(r'=\s*["\'](-?\d+(?:\.\d+)?)["\']', r"= \1", s)
    return s


def semantic_similarity(a: str, b: str) -> float | None:
    if not USE_SEMANTIC:
        return None
    emb = sim_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))


def format_mistral_prompt(schema: str, question: str) -> str:
    user_content = system_message.format(schema=schema) + "\n\n" + question
    # Omit leading "<s>" because llama-cpp-python prepends BOS automatically
    return f"[INST] {user_content} [/INST]"


llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=1024,
    n_gpu_layers=-1,  # set to 0 for CPU-only inference
    verbose=False,
)

exact_hits = 0
norm_hits = 0
sim_scores = []

samples = eval_raw.select(range(N_SAMPLES))

for idx, s in enumerate(samples, start=1):
    prompt = format_mistral_prompt(s["context"], s["question"])
    out = llm(
        prompt,
        max_tokens=128,
        temperature=0.0,
        stop=["</s>"],
    )
    pred = out["choices"][0]["text"].strip()
    expected = s["answer"].strip()

    exact_ok = " ".join(pred.lower().split()) == " ".join(expected.lower().split())
    norm_ok = normalize_sql(pred) == normalize_sql(expected)
    sim = semantic_similarity(pred, expected)

    exact_hits += int(exact_ok)
    norm_hits += int(norm_ok)
    if sim is not None:
        sim_scores.append(sim)

    sim_str = f" | sim: {sim:.4f}" if sim is not None else ""
    if not norm_ok:
        print(
            f"[{idx}/{N_SAMPLES}] Q: {s['question']}\n"
            f"  expected: {expected}\n"
            f"  got:      {pred}\n"
            f"  exact: {exact_ok} | norm: {norm_ok}{sim_str}\n"
        )

print("-" * 72)
print(f"Model:                  {GGUF_PATH}")
print(f"Exact match:            {exact_hits}/{N_SAMPLES}")
print(f"Normalized match:       {norm_hits}/{N_SAMPLES}")
if sim_scores:
    print(f"Mean semantic sim:      {np.mean(sim_scores):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[14/100] Q: Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.
  expected: SELECT name FROM instructor WHERE salary > (SELECT MAX(salary) FROM instructor WHERE dept_name = 'Biology')
  got:      SELECT name FROM instructor WHERE salary > (SELECT MIN(salary) FROM instructor WHERE dep_name = 'BioLogy')
  exact: False | norm: False | sim: 0.7879

[31/100] Q: How many times was the result was re-elected and the party was anti-masonic in the district Pennsylvania 24?
  expected: SELECT COUNT(first_elected) FROM table_2668199_2 WHERE result = "Re-elected" AND party = "Anti-Masonic" AND district = "Pennsylvania 24"
  got:      SELECT COUNT(firsst_elected) FROM table_2668199_2 WHERE result = "Re-elected" AND party = "Anti-Masonic" AND district = "Pennsylvania 24"
  exact: False | norm: False | sim: 0.9657

[33/100] Q: Who's the First District with a Fifth District of prudy adam?
  expected: SELECT first_district FROM table_na

In [3]:
import re
import numpy as np
from datasets import load_dataset
from llama_cpp import Llama

SEED = 123
GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q4_K_M.gguf"
# Or swap to Q8_0:
# GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf"

N_SAMPLES = 100

system_message = (
    "You are a text-to-SQL query translator. Users will ask you questions in "
    "English and you will generate a SQL query based on the provided SCHEMA.\n"
    "SCHEMA:\n{schema}"
)

# Recreate eval_raw identically to the training cell
raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=SEED)
eval_raw = raw.select(range(12500, 13000))

# Optional semantic similarity setup
try:
    from sentence_transformers import SentenceTransformer
    sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    USE_SEMANTIC = True
except ImportError:
    sim_model = None
    USE_SEMANTIC = False


def normalize_sql(sql: str) -> str:
    """Normalize whitespace, casing, and unquote numeric literals for cosmetic comparison."""
    s = " ".join(sql.strip().lower().split())
    s = re.sub(r'=\s*["\'](-?\d+(?:\.\d+)?)["\']', r"= \1", s)
    return s


def semantic_similarity(a: str, b: str) -> float | None:
    if not USE_SEMANTIC:
        return None
    emb = sim_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))


def format_mistral_prompt(schema: str, question: str) -> str:
    user_content = system_message.format(schema=schema) + "\n\n" + question
    return f"[INST] {user_content} [/INST]"


llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=1024,
    n_gpu_layers=-1,  # set to 0 for CPU-only inference
    verbose=False,
)

exact_hits = 0
norm_hits = 0
sim_scores = []

samples = eval_raw.select(range(N_SAMPLES))

for idx, s in enumerate(samples, start=1):
    prompt = format_mistral_prompt(s["context"], s["question"])
    out = llm(
        prompt,
        max_tokens=128,
        temperature=0.0,
        stop=["</s>"],
    )
    pred = out["choices"][0]["text"].strip()
    expected = s["answer"].strip()

    exact_ok = " ".join(pred.lower().split()) == " ".join(expected.lower().split())
    norm_ok = normalize_sql(pred) == normalize_sql(expected)
    sim = semantic_similarity(pred, expected)

    exact_hits += int(exact_ok)
    norm_hits += int(norm_ok)
    if sim is not None:
        sim_scores.append(sim)

    sim_str = f" | sim: {sim:.4f}" if sim is not None else ""
    if not norm_ok:
        print(
            f"[{idx}/{N_SAMPLES}] Q: {s['question']}\n"
            f"  expected: {expected}\n"
            f"  got:      {pred}\n"
            f"  exact: {exact_ok} | norm: {norm_ok}{sim_str}\n"
        )

print("-" * 72)
print(f"Model:                  {GGUF_PATH}")
print(f"Exact match:            {exact_hits}/{N_SAMPLES}")
print(f"Normalized match:       {norm_hits}/{N_SAMPLES}")
if sim_scores:
    print(f"Mean semantic sim:      {np.mean(sim_scores):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[14/100] Q: Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.
  expected: SELECT name FROM instructor WHERE salary > (SELECT MAX(salary) FROM instructor WHERE dept_name = 'Biology')
  got:      SELECT name FROM instructor WHERE salary > (SELECT MIN(salary) FROM instructor WHERE dep_name = 'BioLogy')
  exact: False | norm: False | sim: 0.7879

[31/100] Q: How many times was the result was re-elected and the party was anti-masonic in the district Pennsylvania 24?
  expected: SELECT COUNT(first_elected) FROM table_2668199_2 WHERE result = "Re-elected" AND party = "Anti-Masonic" AND district = "Pennsylvania 24"
  got:      SELECT COUNT(firsst_elected) FROM table_2668199_2 WHERE result = "Re-elected" AND party = "Anti-Masonic" AND district = "Pennsylvania 24"
  exact: False | norm: False | sim: 0.9657

[33/100] Q: Who's the First District with a Fifth District of prudy adam?
  expected: SELECT first_district FROM table_na

In [1]:
import re
import time
import numpy as np
from datasets import load_dataset
from llama_cpp import Llama

# =====================================================================
# 1. CONFIGURATION
# =====================================================================
SEED = 123
GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf"
N_SAMPLES = 100

system_message = (
    "You are a text-to-SQL query translator. Users will ask you questions in "
    "English and you will generate a SQL query based on the provided SCHEMA.\n"
    "SCHEMA:\n{schema}"
)

# Recreate the exact validation slice used during training
raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=SEED)
eval_raw = raw.select(range(12500, 13000))
samples = eval_raw.select(range(N_SAMPLES))

# Optional semantic similarity scorer (runs on CPU so GPU VRAM stays untouched)
try:
    from sentence_transformers import SentenceTransformer
    sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    USE_SEMANTIC = True
except ImportError:
    sim_model = None
    USE_SEMANTIC = False


def normalize_sql(sql: str) -> str:
    """Normalize whitespace, casing, and unquote numeric literals for cosmetic comparison."""
    s = " ".join(sql.strip().lower().split())
    s = re.sub(r'=\s*["\'](-?\d+(?:\.\d+)?)["\']', r"= \1", s)
    return s


def semantic_similarity(a: str, b: str) -> float | None:
    if not USE_SEMANTIC:
        return None
    emb = sim_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))


def format_mistral_prompt(schema: str, question: str) -> str:
    user_content = system_message.format(schema=schema) + "\n\n" + question
    # Omit leading "<s>" because llama-cpp-python prepends BOS automatically
    return f"[INST] {user_content} [/INST]"


# =====================================================================
# 2. LOAD Q8_0 GGUF MODEL
# =====================================================================
llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=1024,
    n_gpu_layers=-1,  # set to 0 for CPU-only inference
    verbose=False,
)

# =====================================================================
# 3. RUN 100-SAMPLE EVALUATION
# =====================================================================
exact_hits = 0
norm_hits = 0
sim_scores = []
total_gen_tokens = 0
t0 = time.perf_counter()

for idx, s in enumerate(samples, start=1):
    prompt = format_mistral_prompt(s["context"], s["question"])
    out = llm(
        prompt,
        max_tokens=128,
        temperature=0.0,
        stop=["</s>"],
    )
    pred = out["choices"][0]["text"].strip()
    expected = s["answer"].strip()
    total_gen_tokens += out["usage"]["completion_tokens"]

    exact_ok = " ".join(pred.lower().split()) == " ".join(expected.lower().split())
    norm_ok = normalize_sql(pred) == normalize_sql(expected)
    sim = semantic_similarity(pred, expected)

    exact_hits += int(exact_ok)
    norm_hits += int(norm_ok)
    if sim is not None:
        sim_scores.append(sim)

    sim_str = f" | sim: {sim:.4f}" if sim is not None else ""
    print(
        f"[{idx:03d}/{N_SAMPLES}] exact: {str(exact_ok):<5} | norm: {str(norm_ok):<5}{sim_str}\n"
        f"  Q:        {s['question']}\n"
        f"  expected: {expected}\n"
        f"  got:      {pred}\n"
    )

elapsed = time.perf_counter() - t0
tok_per_sec = total_gen_tokens / elapsed if elapsed > 0 else 0.0

# =====================================================================
# 4. FINAL SUMMARY REPORT
# =====================================================================
print("=" * 72)
print("FINAL TEST SUMMARY (100 SAMPLES - Q8_0)")
print("=" * 72)
print(f"Checkpoint:             {GGUF_PATH}")
print(f"Exact Match:            {exact_hits}/{N_SAMPLES} ({exact_hits / N_SAMPLES * 100:.1f}%)")
print(f"Normalized Match:       {norm_hits}/{N_SAMPLES} ({norm_hits / N_SAMPLES * 100:.1f}%)")
if sim_scores:
    print(f"Mean Semantic Sim:      {np.mean(sim_scores):.4f} (min: {np.min(sim_scores):.4f})")
print(f"Total Generated Tokens: {total_gen_tokens}")
print(f"Total Elapsed Time:     {elapsed:.2f}s ({tok_per_sec:.1f} tok/s)")
print("=" * 72)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[001/100] exact: True  | norm: True  | sim: 1.0000
  Q:        How many copies per particle are created by the enterotoxin?
  expected: SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"
  got:      SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"

[002/100] exact: False | norm: True  | sim: 0.9916
  Q:        What is the Score of the game of the Player in Place 4?
  expected: SELECT score FROM table_name_16 WHERE place = "4"
  got:      SELECT score FROM table_name_16 WHERE place = 4

[003/100] exact: True  | norm: True  | sim: 1.0000
  Q:        How many heats does Julia Wilkinson have?
  expected: SELECT COUNT(heat) FROM table_name_88 WHERE name = "julia wilkinson"
  got:      SELECT COUNT(heat) FROM table_name_88 WHERE name = "julia wilkinson"

[004/100] exact: True  | norm: True  | sim: 1.0000
  Q:        Which Away captain has a Date of 2,3,4,5 january 1999?
  expected: SELECT away_captain FROM table_name_69 WHERE date = "2,

## HF

In [2]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

user_or_name = 'frankmorales2020'
REPO_ID = f"{user_or_name}/mistral-7b-text-to-sql-topo-Q8_0-GGUF"
GGUF_PATH = "/content/drive/MyDrive/topogpt/mistral-7b-text-to-sql-topo-Q8_0.gguf"

api = HfApi()

api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

api.upload_file(
    path_or_fileobj=GGUF_PATH,
    path_in_repo=os.path.basename(GGUF_PATH),
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"Uploaded {os.path.basename(GGUF_PATH)} to https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ext-to-sql-topo-Q8_0.gguf:   0%|          | 5.32MB / 7.70GB            

Validating                    :           |   0%            

Uploaded mistral-7b-text-to-sql-topo-Q8_0.gguf to https://huggingface.co/frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF


## HF-INFERENCE

In [3]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_ID = "frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF"
FILENAME = "mistral-7b-text-to-sql-topo-Q8_0.gguf"

# Download (or load from local HF cache)
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

# Initialize llama.cpp (set n_gpu_layers=0 for CPU-only inference)
llm = Llama(
    model_path=model_path,
    n_ctx=1024,
    n_gpu_layers=-1,
    verbose=False,
)

system_message = (
    "You are a text-to-SQL query translator. Users will ask you questions in "
    "English and you will generate a SQL query based on the provided SCHEMA.\n"
    "SCHEMA:\n{schema}"
)


def generate_sql(schema: str, question: str) -> str:
    user_content = system_message.format(schema=schema) + "\n\n" + question
    # Omit leading "<s>" because llama-cpp-python prepends BOS automatically
    prompt = f"[INST] {user_content} [/INST]"
    out = llm(
        prompt,
        max_tokens=128,
        temperature=0.0,
        stop=["</s>"],
    )
    return out["choices"][0]["text"].strip()


# Quick smoke tests
tests = [
    {
        "schema": "CREATE TABLE table_140968_1 (copies_per_particle VARCHAR, function VARCHAR)",
        "question": "How many copies per particle are created by the enterotoxin?",
    },
    {
        "schema": "CREATE TABLE Movie (director VARCHAR, title VARCHAR)",
        "question": "What are the names of the directors who made exactly one movie?",
    },
]

for t in tests:
    sql = generate_sql(t["schema"], t["question"])
    print(f"Q:   {t['question']}\nSQL: {sql}\n")

mistral-7b-text-to-sql-topo-Q8_0.gguf: reconstructing file:   0%|          |  0.00B / 7.70GB            

mistral-7b-text-to-sql-topo-Q8_0.gguf: downloading bytes:           |  0.00B            

Q:   How many copies per particle are created by the enterotoxin?
SQL: SELECT copies_per_particle FROM table_140968_1 WHERE function = "Enterotoxin"

Q:   What are the names of the directors who made exactly one movie?
SQL: SELECT director FROM Movie GROUP BY director HAVING COUNT(*) = 1



In [4]:
import re
import time
import numpy as np
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# =====================================================================
# 1. CONFIGURATION & HF DOWNLOAD
# =====================================================================
SEED = 123
REPO_ID = "frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF"
FILENAME = "mistral-7b-text-to-sql-topo-Q8_0.gguf"
N_SAMPLES = 100

model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

system_message = (
    "You are a text-to-SQL query translator. Users will ask you questions in "
    "English and you will generate a SQL query based on the provided SCHEMA.\n"
    "SCHEMA:\n{schema}"
)

# Recreate the exact validation slice used during training
raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=SEED)
eval_raw = raw.select(range(12500, 13000))
samples = eval_raw.select(range(N_SAMPLES))

# Optional semantic similarity scorer on CPU
try:
    from sentence_transformers import SentenceTransformer
    sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    USE_SEMANTIC = True
except ImportError:
    sim_model = None
    USE_SEMANTIC = False


def normalize_sql(sql: str) -> str:
    """Normalize whitespace, casing, and unquote numeric literals."""
    s = " ".join(sql.strip().lower().split())
    s = re.sub(r'=\s*["\'](-?\d+(?:\.\d+)?)["\']', r"= \1", s)
    return s


def semantic_similarity(a: str, b: str) -> float | None:
    if not USE_SEMANTIC:
        return None
    emb = sim_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))


def format_mistral_prompt(schema: str, question: str) -> str:
    user_content = system_message.format(schema=schema) + "\n\n" + question
    # Omit leading "<s>" because llama-cpp-python prepends BOS automatically
    return f"[INST] {user_content} [/INST]"


# =====================================================================
# 2. LOAD MODEL FROM HUGGING FACE CACHE
# =====================================================================
llm = Llama(
    model_path=model_path,
    n_ctx=1024,
    n_gpu_layers=-1,  # set to 0 for CPU-only inference
    verbose=False,
)

# =====================================================================
# 3. RUN 100-QUESTION INFERENCE BENCHMARK
# =====================================================================
exact_hits = 0
norm_hits = 0
sim_scores = []
total_gen_tokens = 0
t0 = time.perf_counter()

for idx, s in enumerate(samples, start=1):
    prompt = format_mistral_prompt(s["context"], s["question"])
    out = llm(
        prompt,
        max_tokens=128,
        temperature=0.0,
        stop=["</s>"],
    )
    pred = out["choices"][0]["text"].strip()
    expected = s["answer"].strip()
    total_gen_tokens += out["usage"]["completion_tokens"]

    exact_ok = " ".join(pred.lower().split()) == " ".join(expected.lower().split())
    norm_ok = normalize_sql(pred) == normalize_sql(expected)
    sim = semantic_similarity(pred, expected)

    exact_hits += int(exact_ok)
    norm_hits += int(norm_ok)
    if sim is not None:
        sim_scores.append(sim)

    sim_str = f" | sim: {sim:.4f}" if sim is not None else ""
    if not norm_ok:
        print(
            f"[{idx:03d}/{N_SAMPLES}] exact: {str(exact_ok):<5} | norm: {str(norm_ok):<5}{sim_str}\n"
            f"  Q:        {s['question']}\n"
            f"  expected: {expected}\n"
            f"  got:      {pred}\n"
        )

elapsed = time.perf_counter() - t0
tok_per_sec = total_gen_tokens / elapsed if elapsed > 0 else 0.0

# =====================================================================
# 4. SUMMARY REPORT
# =====================================================================
print("=" * 72)
print("HUGGING FACE Q8_0 INFERENCE BENCHMARK (100 QUESTIONS)")
print("=" * 72)
print(f"Repo:                   {REPO_ID}")
print(f"File:                   {FILENAME}")
print(f"Exact Match:            {exact_hits}/{N_SAMPLES} ({exact_hits / N_SAMPLES * 100:.1f}%)")
print(f"Normalized Match:       {norm_hits}/{N_SAMPLES} ({norm_hits / N_SAMPLES * 100:.1f}%)")
if sim_scores:
    print(f"Mean Semantic Sim:      {np.mean(sim_scores):.4f} (min: {np.min(sim_scores):.4f})")
print(f"Total Generated Tokens: {total_gen_tokens}")
print(f"Total Elapsed Time:     {elapsed:.2f}s ({tok_per_sec:.1f} tok/s)")
print("=" * 72)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[014/100] exact: False | norm: False | sim: 0.7879
  Q:        Find the names of all instructors whose salary is greater than the salary of all instructors in the Biology department.
  expected: SELECT name FROM instructor WHERE salary > (SELECT MAX(salary) FROM instructor WHERE dept_name = 'Biology')
  got:      SELECT name FROM instructor WHERE salary > (SELECT MIN(salary) FROM instructor WHERE dep_name = 'BioLogy')

[031/100] exact: False | norm: False | sim: 0.9657
  Q:        How many times was the result was re-elected and the party was anti-masonic in the district Pennsylvania 24?
  expected: SELECT COUNT(first_elected) FROM table_2668199_2 WHERE result = "Re-elected" AND party = "Anti-Masonic" AND district = "Pennsylvania 24"
  got:      SELECT COUNT(firsst_elected) FROM table_2668199_2 WHERE result = "Re-elected" AND party = "Anti-Masonic" AND district = "Pennsylvania 24"

[033/100] exact: False | norm: False | sim: 0.7104
  Q:        Who's the First District with a Fifth Dist